# Traffic Sign Detection with YOLOv5

Trains a YOLOv5s detector on 13 classes of road signs and runs it on the held-out test images.

Designed for Google Colab with a GPU runtime (**Runtime > Change runtime type > GPU**). The dataset is read from Google Drive as a zip containing `train/`, `valid/` and `test/` folders, each with `images/` and `labels/` in YOLO format.

## 1. Configuration

In [ ]:
# Paths on Google Drive. Adjust these to match where you keep the dataset and want results saved.
DATASET_ZIP = '/content/drive/MyDrive/Yolov5/trafficSign.zip'
SAVE_DIR = '/content/drive/MyDrive/yolo5Saved'

DATA_DIR = '/content'          # the zip is extracted here (train/, valid/, test/)
YOLO_DIR = '/content/yolov5'
YOLO_VERSION = 'v7.0'         # pinned YOLOv5 release for reproducible behaviour

CLASS_NAMES = ['SLimit30', 'SLimit40', 'SLimit50', 'RoundAbout', 'NoEnter', 'Stop', 'DriveR',
               'DriveL', 'CrossWay', 'GoStraight', 'Yield', 'DontStop', 'Parking']

# Training settings used for the results in this repository.
IMG_SIZE = 1216               # rounded up to a multiple of 32 by YOLOv5 (1200 was passed originally)
BATCH = 20
EPOCHS = 50
WEIGHTS = "''"                # train from scratch; use 'yolov5s.pt' to fine-tune from COCO weights instead

# Inference settings.
DETECT_IMG_SIZE = 650
CONF_THRESHOLD = 0.2

## 2. Set up YOLOv5

In [ ]:
!git clone --depth 1 --branch {YOLO_VERSION} https://github.com/ultralytics/yolov5 {YOLO_DIR}
!pip install -q -r {YOLO_DIR}/requirements.txt

In [ ]:
import torch

device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'torch {torch.__version__} on {device}')

## 3. Load the dataset

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
!unzip -q -o "{DATASET_ZIP}" -d "{DATA_DIR}"
!ls {DATA_DIR}/train/images | wc -l; ls {DATA_DIR}/valid/images | wc -l; ls {DATA_DIR}/test/images | wc -l

Write the dataset description that YOLOv5 reads during training and evaluation.

In [ ]:
import yaml

DATA_YAML = f'{YOLO_DIR}/data/traffic_signs.yaml'
data_config = {
    'train': f'{DATA_DIR}/train/images',
    'val': f'{DATA_DIR}/valid/images',
    'test': f'{DATA_DIR}/test/images',
    'nc': len(CLASS_NAMES),
    'names': CLASS_NAMES,
}
with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print(open(DATA_YAML).read())

## 4. Model definition

YOLOv5s layout (depth 0.33, width 0.50) using `BottleneckCSP` blocks, with the detection head sized for our class count.

In [ ]:
MODEL_YAML = f'{YOLO_DIR}/models/custom_yolov5s.yaml'

model_config = f'''
nc: {len(CLASS_NAMES)}  # number of classes
depth_multiple: 0.33  # model depth multiple
width_multiple: 0.50  # layer channel multiple

anchors:
  - [10,13, 16,30, 33,23]  # P3/8
  - [30,61, 62,45, 59,119]  # P4/16
  - [116,90, 156,198, 373,326]  # P5/32

backbone:
  # [from, number, module, args]
  [[-1, 1, Focus, [64, 3]],  # 0-P1/2
   [-1, 1, Conv, [128, 3, 2]],  # 1-P2/4
   [-1, 3, BottleneckCSP, [128]],
   [-1, 1, Conv, [256, 3, 2]],  # 3-P3/8
   [-1, 9, BottleneckCSP, [256]],
   [-1, 1, Conv, [512, 3, 2]],  # 5-P4/16
   [-1, 9, BottleneckCSP, [512]],
   [-1, 1, Conv, [1024, 3, 2]],  # 7-P5/32
   [-1, 1, SPP, [1024, [5, 9, 13]]],
   [-1, 3, BottleneckCSP, [1024, False]],  # 9
  ]

head:
  [[-1, 1, Conv, [512, 1, 1]],
   [-1, 1, nn.Upsample, [None, 2, 'nearest']],
   [[-1, 6], 1, Concat, [1]],  # cat backbone P4
   [-1, 3, BottleneckCSP, [512, False]],  # 13

   [-1, 1, Conv, [256, 1, 1]],
   [-1, 1, nn.Upsample, [None, 2, 'nearest']],
   [[-1, 4], 1, Concat, [1]],  # cat backbone P3
   [-1, 3, BottleneckCSP, [256, False]],  # 17 (P3/8-small)

   [-1, 1, Conv, [256, 3, 2]],
   [[-1, 14], 1, Concat, [1]],  # cat head P4
   [-1, 3, BottleneckCSP, [512, False]],  # 20 (P4/16-medium)

   [-1, 1, Conv, [512, 3, 2]],
   [[-1, 10], 1, Concat, [1]],  # cat head P5
   [-1, 3, BottleneckCSP, [1024, False]],  # 23 (P5/32-large)

   [[17, 20, 23], 1, Detect, [nc, anchors]],  # Detect(P3, P4, P5)
  ]
'''
with open(MODEL_YAML, 'w') as f:
    f.write(model_config)

## 5. Train

In [ ]:
%cd {YOLO_DIR}
!python train.py --img {IMG_SIZE} --batch {BATCH} --epochs {EPOCHS} \
    --data {DATA_YAML} --cfg {MODEL_YAML} --weights {WEIGHTS} \
    --project "{SAVE_DIR}" --name exp --exist-ok

## 6. Inspect training results

Training curves, then a validation batch with ground truth labels next to the model's predictions.

In [ ]:
from IPython.display import Image, display

RUN_DIR = f'{SAVE_DIR}/exp'
display(Image(filename=f'{RUN_DIR}/results.png', width=1000))

In [ ]:
print('Ground truth:')
display(Image(filename=f'{RUN_DIR}/val_batch2_labels.jpg', width=900))
print('Predictions:')
display(Image(filename=f'{RUN_DIR}/val_batch2_pred.jpg', width=900))

## 7. Evaluate on the test split

In [ ]:
!python val.py --weights "{RUN_DIR}/weights/best.pt" --data {DATA_YAML} --img {IMG_SIZE} --task test \
    --project "{SAVE_DIR}" --name test --exist-ok

## 8. Detect signs in the test images

In [ ]:
!python detect.py --weights "{RUN_DIR}/weights/best.pt" --img {DETECT_IMG_SIZE} --conf {CONF_THRESHOLD} \
    --source {DATA_DIR}/test/images --project "{SAVE_DIR}" --name detect --exist-ok

In [ ]:
import glob

for path in sorted(glob.glob(f'{SAVE_DIR}/detect/*.jpg'))[:5]:
    display(Image(filename=path, width=800))